# Анализ `screening_with_trades`

Цель ноутбука: смотреть арбитраж между Steam и CSFloat с учетом того, что после покупки в Steam предмет сидит в **7-дневном бане**. Поэтому фильтры здесь не про "красивый датасет", а про то, чтобы:

- не заходить в слишком тонкие и шумные позиции
- не переоценивать большой spread, если у Steam по истории есть плохой downside
- сохранить удобный просмотр и **больших**, и **маленьких** spread'ов

Структура намеренно простая:
1. импорт и препроцесс
2. risk-фильтры
3. display risk-passed выборки по `spread_pred%` сначала по убыванию, потом по возрастанию


## Risk Metrics Cheatsheet

- `steam_sales_7d_iqr_risk% = (p75 - p25) / mean * 100`
  Центральный разброс сделок за 7 дней. Больше = шире нормальный диапазон цен.

- `steam_sales_7d_band_risk% = (p90 - p10) / mean * 100`
  Широкий диапазон сделок за 7 дней. Больше = выше общая турбулентность.

- `steam_sales_7d_downside_risk% = (median - p10) / mean * 100`
  Насколько нижний хвост уходит вниз от типичной цены. Больше = хуже downside.

- `steam_sales_7d_tail_ratio = p10 / median`
  Насколько низ хвоста близок к медиане. Ближе к `1` = лучше.

- `steam_daily_ret_3d = current_daily_median / daily_median_3d_ago - 1`
  Изменение дневной медианы за 3 дня. Ниже = слабее краткосрок.

- `steam_daily_ret_7d = current_daily_median / daily_median_7d_ago - 1`
  Изменение дневной медианы за 7 дней. Ниже = слабее фон на горизонте холда.

- `steam_daily_slope_7d = slope(log(daily_median))`
  Наклон лог-цены за 7 дней. Ниже = слабее устойчивый тренд.

- `steam_daily_ema_gap_3_14 = EMA(3) / EMA(14) - 1`
  Короткая EMA против длинной. Ниже `0` = локальный momentum слабый.

- `steam_daily_range_14d_pct = (max_14d - min_14d) / current_daily_median`
  Полный диапазон дневной медианы за 14 дней. Больше = шире среда.

- `steam_daily_downside_14d_pct = (current_daily_median - min_14d) / current_daily_median`
  Насколько текущая цена выше недавнего 14-day low. Больше = под текущей ценой уже был более глубокий низ.

- `steam_sales_7d_n`
  Число сделок в weekly summary. Больше = risk-метрики надежнее.


In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

USE_LATEST_SCREENING = True
SCREENING_GLOBS = ("screening_full_trades_*.csv", "screening_sub_trades_*.csv")


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data_with_trades").is_dir():
        return cwd
    if (cwd.parent / "data_with_trades").is_dir():
        return cwd.parent
    raise FileNotFoundError(
        "Не удалось найти папку data_with_trades ни в cwd, ни уровнем выше"
    )


PROJECT_ROOT = resolve_project_root()
SCREENING_DATA_DIR = PROJECT_ROOT / "data_with_trades"
CSV_FILE_MANUAL = SCREENING_DATA_DIR / "screening_full_trades_20260422_044314.csv"

ITEM_INCLUDE_ANY: list[str] = []
ITEM_EXCLUDE_ANY: list[str] = []
TOP_N = 50

RISK_FILTERS = {
    "steam_ask_min": 1.25,
    "steam_ask_max": 80.0,
    "float_qty_min": 30,
    "steam_sales_n_min": 50,
    "downside_risk_max": 10.0,
    "tail_ratio_min": 0.90,
    "downside_14d_max": 0.12,
}


def resolve_csv() -> Path:
    if not USE_LATEST_SCREENING:
        return CSV_FILE_MANUAL.resolve()

    found: list[Path] = []
    seen: set[Path] = set()
    for pattern in SCREENING_GLOBS:
        for path in SCREENING_DATA_DIR.glob(pattern):
            rp = path.resolve()
            if rp not in seen:
                seen.add(rp)
                found.append(path)
    found.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if not found:
        raise FileNotFoundError(
            f"Нет файлов {SCREENING_GLOBS} в {SCREENING_DATA_DIR.resolve()}"
        )
    return found[0].resolve()


def contains_any(series: pd.Series, needles: list[str]) -> pd.Series:
    if not needles:
        return pd.Series(True, index=series.index)
    pattern = "|".join(map(str, needles))
    return series.str.contains(pattern, case=False, na=False, regex=True)


In [2]:
csv_file = resolve_csv()
df = pd.read_csv(csv_file)

for col in df.columns:
    if col != "item":
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.copy()
df["tail_gap%"] = (1.0 - df["steam_sales_7d_tail_ratio"]) * 100.0

core_cols = [
    "item",
    "steam_ask",
    "float_ask",
    "float_pred",
    "float_qty",
    "spread_ask%",
    "spread_pred%",
    "steam_sales_7d_n",
    "steam_sales_7d_iqr_risk%",
    "steam_sales_7d_band_risk%",
    "steam_sales_7d_downside_risk%",
    "steam_sales_7d_tail_ratio",
    "tail_gap%",
    "steam_daily_ret_3d",
    "steam_daily_ret_7d",
    "steam_daily_slope_7d",
    "steam_daily_ema_gap_3_14",
    "steam_daily_range_14d_pct",
    "steam_daily_downside_14d_pct",
]

print("CSV_FILE =", csv_file)
print(f"rows={len(df)}, cols={len(df.columns)}")
display(df[core_cols].head(10))


CSV_FILE = C:\Roman\skins_roundtrip_v1\data_with_trades\screening_full_trades_20260424_234219.csv
rows=4, cols=29


,item,steam_ask,float_ask,float_pred,float_qty,spread_ask%,spread_pred%,steam_sales_7d_n,steam_sales_7d_iqr_risk%,steam_sales_7d_band_risk%,steam_sales_7d_downside_risk%,steam_sales_7d_tail_ratio,tail_gap%,steam_daily_ret_3d,steam_daily_ret_7d,steam_daily_slope_7d,steam_daily_ema_gap_3_14,steam_daily_range_14d_pct,steam_daily_downside_14d_pct
0,"Music Kit | Selective Response, No Love Only Pleasure",2.48,1.75,1.31,145,29.42,46.98,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sealed Dead Hand Terminal,2.96,1.79,1.72,4753,39.42,42.02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Glock-18 | Catacombs (Field-Tested),1.57,1.10,1.09,140,29.84,30.39,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Glock-18 | Catacombs (Minimal Wear),1.79,1.28,1.09,97,28.45,38.94,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Risk Filters

Ниже только **risk / liquidity gate**. Спред здесь **не фильтруется**, чтобы после прохождения риска можно было отдельно посмотреть и лучшие большие spread'ы, и самые слабые маленькие.


In [3]:
filter_specs: list[tuple[str, pd.Series]] = [
    (
        f"steam_ask in [{RISK_FILTERS['steam_ask_min']}, {RISK_FILTERS['steam_ask_max']}]",
        df["steam_ask"].between(RISK_FILTERS["steam_ask_min"], RISK_FILTERS["steam_ask_max"]),
    ),
    (
        f"float_qty >= {RISK_FILTERS['float_qty_min']}",
        df["float_qty"] >= RISK_FILTERS["float_qty_min"],
    ),
    (
        f"steam_sales_7d_n >= {RISK_FILTERS['steam_sales_n_min']}",
        df["steam_sales_7d_n"] >= RISK_FILTERS["steam_sales_n_min"],
    ),
    (
        f"steam_sales_7d_downside_risk% <= {RISK_FILTERS['downside_risk_max']}",
        df["steam_sales_7d_downside_risk%"] <= RISK_FILTERS["downside_risk_max"],
    ),
    (
        f"steam_sales_7d_tail_ratio >= {RISK_FILTERS['tail_ratio_min']}",
        df["steam_sales_7d_tail_ratio"] >= RISK_FILTERS["tail_ratio_min"],
    ),
    (
        f"steam_daily_downside_14d_pct <= {RISK_FILTERS['downside_14d_max']}",
        df["steam_daily_downside_14d_pct"] <= RISK_FILTERS["downside_14d_max"],
    ),
]

if ITEM_INCLUDE_ANY:
    filter_specs.append((f"include_any={ITEM_INCLUDE_ANY}", contains_any(df["item"], ITEM_INCLUDE_ANY)))
if ITEM_EXCLUDE_ANY:
    filter_specs.append((f"exclude_any={ITEM_EXCLUDE_ANY}", ~contains_any(df["item"], ITEM_EXCLUDE_ANY)))

risk_masks = {label: mask.fillna(False).astype(bool) for label, mask in filter_specs}
risk_pass = pd.Series(True, index=df.index)
for mask in risk_masks.values():
    risk_pass &= mask

fail_matrix = pd.DataFrame({label: ~mask for label, mask in risk_masks.items()}, index=df.index)
df["risk_pass"] = risk_pass
df["risk_fail_count"] = fail_matrix.sum(axis=1)
df["risk_fail_reasons"] = fail_matrix.apply(
    lambda row: ", ".join([col for col, failed in row.items() if failed]) if row.any() else "-",
    axis=1,
)

risk_report = pd.DataFrame(
    [
        {
            "rule": label,
            "passed_rows": int(mask.sum()),
            "failed_rows": int((~mask).sum()),
        }
        for label, mask in risk_masks.items()
    ]
)

risk_df = df.loc[df["risk_pass"]].copy()
failed_df = df.loc[~df["risk_pass"]].copy()

print(f"{len(df)} loaded -> {len(risk_df)} passed risk filters -> {len(failed_df)} failed")
display(risk_report)
display(
    failed_df[
        [
            "item",
            "spread_pred%",
            "steam_sales_7d_n",
            "steam_sales_7d_band_risk%",
            "steam_sales_7d_downside_risk%",
            "steam_sales_7d_tail_ratio",
            "steam_daily_downside_14d_pct",
            "risk_fail_count",
            "risk_fail_reasons",
        ]
    ]
    .sort_values(["risk_fail_count", "spread_pred%"], ascending=[True, False])
    .head(20)
    .reset_index(drop=True)
)


4 loaded -> 0 passed risk filters -> 4 failed


,rule,passed_rows,failed_rows
0,"steam_ask in [1.25, 80.0]",4,0
1,float_qty >= 30,4,0
2,steam_sales_7d_n >= 50,0,4
3,steam_sales_7d_downside_risk% <= 10.0,0,4
4,steam_sales_7d_tail_ratio >= 0.9,0,4
5,steam_daily_downside_14d_pct <= 0.12,0,4


,item,spread_pred%,steam_sales_7d_n,steam_sales_7d_band_risk%,steam_sales_7d_downside_risk%,steam_sales_7d_tail_ratio,steam_daily_downside_14d_pct,risk_fail_count,risk_fail_reasons
0,"Music Kit | Selective Response, No Love Only Pleasure",46.98,0,NaN,NaN,NaN,NaN,4,"steam_sales_7d_n >= 50, steam_sales_7d_downside_risk% <= 10.0, steam_sales_7d_tail_ratio >= 0.9, steam_daily_downsid..."
1,Sealed Dead Hand Terminal,42.02,0,NaN,NaN,NaN,NaN,4,"steam_sales_7d_n >= 50, steam_sales_7d_downside_risk% <= 10.0, steam_sales_7d_tail_ratio >= 0.9, steam_daily_downsid..."
2,Glock-18 | Catacombs (Minimal Wear),38.94,0,NaN,NaN,NaN,NaN,4,"steam_sales_7d_n >= 50, steam_sales_7d_downside_risk% <= 10.0, steam_sales_7d_tail_ratio >= 0.9, steam_daily_downsid..."
3,Glock-18 | Catacombs (Field-Tested),30.39,0,NaN,NaN,NaN,NaN,4,"steam_sales_7d_n >= 50, steam_sales_7d_downside_risk% <= 10.0, steam_sales_7d_tail_ratio >= 0.9, steam_daily_downsid..."


## Display By Spread

Одна и та же `risk_df` показывается в двух срезах:
- сначала где spread самый большой
- потом где spread самый маленький


In [4]:
display_cols = [
    "item",
    "steam_ask",
    "float_ask",
    "float_pred",
    "float_qty",
    "spread_ask%",
    "spread_pred%",
    "steam_sales_7d_n",
    "steam_sales_7d_iqr_risk%",
    "steam_sales_7d_band_risk%",
    "steam_sales_7d_downside_risk%",
    "steam_sales_7d_tail_ratio",
    "tail_gap%",
    "steam_daily_ret_3d",
    "steam_daily_ret_7d",
    "steam_daily_slope_7d",
    "steam_daily_ema_gap_3_14",
    "steam_daily_range_14d_pct",
    "steam_daily_downside_14d_pct",
    "risk_fail_reasons",
]

LEVEL_COLORS = {
    "excellent": "#1f7a1f",
    "very_good": "#5dbb63",
    "good": "#a9d18e",
    "ok": "#f2e6a7",
    "bad": "#e89b73",
    "awful": "#d65f5f",
}

COLOR_GRID = {
    "steam_sales_7d_n": {
        "bands": [
            {"label": "awful", "lo": None, "hi": 20},
            {"label": "bad", "lo": 20, "hi": 50},
            {"label": "ok", "lo": 50, "hi": 100},
            {"label": "good", "lo": 100, "hi": 200},
            {"label": "very_good", "lo": 200, "hi": 400},
            {"label": "excellent", "lo": 400, "hi": None},
        ],
    },
    "steam_sales_7d_iqr_risk%": {
        "bands": [
            {"label": "excellent", "lo": None, "hi": 4},
            {"label": "very_good", "lo": 4, "hi": 7},
            {"label": "good", "lo": 7, "hi": 10},
            {"label": "ok", "lo": 10, "hi": 14},
            {"label": "bad", "lo": 14, "hi": 20},
            {"label": "awful", "lo": 20, "hi": None},
        ],
    },
    "steam_sales_7d_band_risk%": {
        "bands": [
            {"label": "excellent", "lo": None, "hi": 8},
            {"label": "very_good", "lo": 8, "hi": 14},
            {"label": "good", "lo": 14, "hi": 22},
            {"label": "ok", "lo": 22, "hi": 32},
            {"label": "bad", "lo": 32, "hi": 45},
            {"label": "awful", "lo": 45, "hi": None},
        ],
    },
    "steam_sales_7d_downside_risk%": {
        "bands": [
            {"label": "excellent", "lo": None, "hi": 2.5},
            {"label": "very_good", "lo": 2.5, "hi": 4.0},
            {"label": "good", "lo": 4.0, "hi": 6.0},
            {"label": "ok", "lo": 6.0, "hi": 8.0},
            {"label": "bad", "lo": 8.0, "hi": 12.0},
            {"label": "awful", "lo": 12.0, "hi": None},
        ],
    },
    "steam_sales_7d_tail_ratio": {
        "bands": [
            {"label": "awful", "lo": None, "hi": 0.85},
            {"label": "bad", "lo": 0.85, "hi": 0.90},
            {"label": "ok", "lo": 0.90, "hi": 0.93},
            {"label": "good", "lo": 0.93, "hi": 0.95},
            {"label": "very_good", "lo": 0.95, "hi": 0.97},
            {"label": "excellent", "lo": 0.97, "hi": None},
        ],
    },
    "steam_daily_ret_3d": {
        "bands": [
            {"label": "awful", "lo": None, "hi": -0.08},
            {"label": "bad", "lo": -0.08, "hi": -0.03},
            {"label": "ok", "lo": -0.03, "hi": 0.0},
            {"label": "good", "lo": 0.0, "hi": 0.03},
            {"label": "very_good", "lo": 0.03, "hi": 0.07},
            {"label": "excellent", "lo": 0.07, "hi": None},
        ],
    },
    "steam_daily_ret_7d": {
        "bands": [
            {"label": "awful", "lo": None, "hi": -0.12},
            {"label": "bad", "lo": -0.12, "hi": -0.05},
            {"label": "ok", "lo": -0.05, "hi": 0.0},
            {"label": "good", "lo": 0.0, "hi": 0.04},
            {"label": "very_good", "lo": 0.04, "hi": 0.10},
            {"label": "excellent", "lo": 0.10, "hi": None},
        ],
    },
    "steam_daily_slope_7d": {
        "bands": [
            {"label": "awful", "lo": None, "hi": -0.02},
            {"label": "bad", "lo": -0.02, "hi": -0.005},
            {"label": "ok", "lo": -0.005, "hi": 0.005},
            {"label": "good", "lo": 0.005, "hi": 0.015},
            {"label": "very_good", "lo": 0.015, "hi": 0.03},
            {"label": "excellent", "lo": 0.03, "hi": None},
        ],
    },
    "steam_daily_ema_gap_3_14": {
        "bands": [
            {"label": "awful", "lo": None, "hi": -0.05},
            {"label": "bad", "lo": -0.05, "hi": -0.015},
            {"label": "ok", "lo": -0.015, "hi": 0.015},
            {"label": "good", "lo": 0.015, "hi": 0.04},
            {"label": "very_good", "lo": 0.04, "hi": 0.08},
            {"label": "excellent", "lo": 0.08, "hi": None},
        ],
    },
    "steam_daily_range_14d_pct": {
        "bands": [
            {"label": "excellent", "lo": None, "hi": 0.04},
            {"label": "very_good", "lo": 0.04, "hi": 0.07},
            {"label": "good", "lo": 0.07, "hi": 0.11},
            {"label": "ok", "lo": 0.11, "hi": 0.16},
            {"label": "bad", "lo": 0.16, "hi": 0.24},
            {"label": "awful", "lo": 0.24, "hi": None},
        ],
    },
    "steam_daily_downside_14d_pct": {
        "bands": [
            {"label": "excellent", "lo": None, "hi": 0.02},
            {"label": "very_good", "lo": 0.02, "hi": 0.05},
            {"label": "good", "lo": 0.05, "hi": 0.08},
            {"label": "ok", "lo": 0.08, "hi": 0.12},
            {"label": "bad", "lo": 0.12, "hi": 0.18},
            {"label": "awful", "lo": 0.18, "hi": None},
        ],
    },
}

FORMATTERS = {
    "steam_ask": "{:.2f}",
    "float_ask": "{:.2f}",
    "float_pred": "{:.2f}",
    "float_qty": "{:.0f}",
    "spread_ask%": "{:.2f}",
    "spread_pred%": "{:.2f}",
    "steam_sales_7d_n": "{:.0f}",
    "steam_sales_7d_iqr_risk%": "{:.2f}",
    "steam_sales_7d_band_risk%": "{:.2f}",
    "steam_sales_7d_downside_risk%": "{:.2f}",
    "steam_sales_7d_tail_ratio": "{:.4f}",
    "tail_gap%": "{:.2f}",
    "steam_daily_ret_3d": "{:.2%}",
    "steam_daily_ret_7d": "{:.2%}",
    "steam_daily_slope_7d": "{:.4f}",
    "steam_daily_ema_gap_3_14": "{:.2%}",
    "steam_daily_range_14d_pct": "{:.2%}",
    "steam_daily_downside_14d_pct": "{:.2%}",
}


def _in_band(value: float, lo: float | None, hi: float | None) -> bool:
    if pd.isna(value):
        return False
    if lo is not None and value < lo:
        return False
    if hi is not None and value >= hi:
        return False
    return True


def _band_color(value: float, spec: dict[str, object]) -> str | None:
    for band in spec.get("bands", []):
        if _in_band(value, band.get("lo"), band.get("hi")):
            return LEVEL_COLORS.get(str(band.get("label")), "")
    return None


def make_fixed_grid_styler(frame: pd.DataFrame):
    def style_series(s: pd.Series):
        spec = COLOR_GRID.get(s.name)
        if spec is None or not pd.api.types.is_numeric_dtype(s):
            return [""] * len(s)
        out = []
        for v in pd.to_numeric(s, errors="coerce"):
            color = _band_color(v, spec)
            if not color:
                out.append("")
                continue
            out.append(f"background-color: {color}; color: black;")
        return out

    formatters = {k: v for k, v in FORMATTERS.items() if k in frame.columns}
    return frame.style.format(formatters, na_rep="-").apply(style_series, axis=0)


print(f"TOP {TOP_N} risk-passed rows by spread_pred% DESC")
top_desc = (
    risk_df[display_cols]
    .sort_values(["spread_pred%", "spread_ask%"], ascending=[False, False])
    .head(TOP_N)
    .reset_index(drop=True)
)
display(make_fixed_grid_styler(top_desc))


TOP 50 risk-passed rows by spread_pred% DESC


,item,steam_ask,float_ask,float_pred,float_qty,spread_ask%,spread_pred%,steam_sales_7d_n,steam_sales_7d_iqr_risk%,steam_sales_7d_band_risk%,steam_sales_7d_downside_risk%,steam_sales_7d_tail_ratio,tail_gap%,steam_daily_ret_3d,steam_daily_ret_7d,steam_daily_slope_7d,steam_daily_ema_gap_3_14,steam_daily_range_14d_pct,steam_daily_downside_14d_pct,risk_fail_reasons


In [5]:
print(f"TOP {TOP_N} risk-passed rows by spread_pred% ASC")
top_asc = (
    risk_df[display_cols]
    .sort_values(["spread_pred%", "spread_ask%"], ascending=[True, True])
    .head(TOP_N)
    .reset_index(drop=True)
)
display(make_fixed_grid_styler(top_asc))


TOP 50 risk-passed rows by spread_pred% ASC


,item,steam_ask,float_ask,float_pred,float_qty,spread_ask%,spread_pred%,steam_sales_7d_n,steam_sales_7d_iqr_risk%,steam_sales_7d_band_risk%,steam_sales_7d_downside_risk%,steam_sales_7d_tail_ratio,tail_gap%,steam_daily_ret_3d,steam_daily_ret_7d,steam_daily_slope_7d,steam_daily_ema_gap_3_14,steam_daily_range_14d_pct,steam_daily_downside_14d_pct,risk_fail_reasons


## Breakeven Spread

Пара A→B:
- на ноге **A** покупаем в Steam и потом выходим в Float
- на ноге **B** покупаем в Float и потом выходим в Steam

Если нога A слабая и дает маленький или отрицательный edge, можно посчитать, какой **минимальный spread на B** нужен, чтобы вся связка хотя бы вышла в ноль после комиссий.


In [8]:
STEAM_FEE = 0.05
FLOAT_FEE = 0.02

A_SPREAD_INPUTS = [0, 5, 10, 15, 20, 25, 27]


def leg_a_after_fees_from_spread(spread_a_pct: float, float_fee: float = FLOAT_FEE) -> float:
    ratio_a = 1.0 - (spread_a_pct / 100.0)
    return ratio_a * (1.0 - float_fee)


def breakeven_b_spread_from_a(
    spread_a_pct: float,
    *,
    steam_fee: float = STEAM_FEE,
    float_fee: float = FLOAT_FEE,
) -> float | None:
    leg_a = leg_a_after_fees_from_spread(spread_a_pct, float_fee=float_fee)
    if leg_a <= 0:
        return None
    required_ratio_b = 1.0 / (leg_a * (1.0 - steam_fee))
    if required_ratio_b <= 0:
        return None
    return (1.0 - (1.0 / required_ratio_b)) * 100.0


rows = []
for spread_a in A_SPREAD_INPUTS:
    leg_a = leg_a_after_fees_from_spread(spread_a)
    need_b = breakeven_b_spread_from_a(spread_a)
    rows.append(
        {
            "A_spread%": spread_a,
            "A_leg_after_float_fee": round(leg_a, 4),
            "min_B_spread%_for_breakeven": round(need_b, 2) if need_b is not None else None,
        }
    )

breakeven_df = pd.DataFrame(rows)
display(breakeven_df)


,A_spread%,A_leg_after_float_fee,min_B_spread%_for_breakeven
0,0,0.9800,6.90
1,5,0.9310,11.56
2,10,0.8820,16.21
3,15,0.8330,20.87
4,20,0.7840,25.52
5,25,0.7350,30.18
6,27,0.7154,32.04
